In [2]:
import cv2
import random
import pandas as pd
from ultralytics import YOLO
import joblib
from collections import deque
import os

# Загружаем модели прогноза (если есть)
model_30 = joblib.load('model_30.pkl') if os.path.exists('model_30.pkl') else None
model_60 = joblib.load('model_60.pkl') if os.path.exists('model_60.pkl') else None
model_120 = joblib.load('model_120.pkl') if os.path.exists('model_120.pkl') else None

# Для прогноза
history = deque(maxlen=30)
last_minute = -1
predictions = []
cars_in_minute = 0

yolo = YOLO("yolo12s.pt")

def getColours(cls_num):
    random.seed(cls_num)
    return tuple(random.randint(0, 255) for _ in range(3))

video_path = "csv/test_traffic.mp4"
videoCap = cv2.VideoCapture(video_path)
fps = videoCap.get(cv2.CAP_PROP_FPS)

frame_count = 0
detections = []

while True:
    ret, frame = videoCap.read()
    if not ret:
        break

    results = yolo.track(frame, stream=True)

    for result in results:
        class_names = result.names
        for box in result.boxes:
            if box.conf[0] > 0.4:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cls = int(box.cls[0])
                class_name = class_names[cls]
                conf = float(box.conf[0])
                track_id = int(box.id[0]) if box.id is not None else -1
                time_ms = frame_count * (1000 / fps)
                
                # Считаем машины в минуту
                cars_in_minute += 1

                detections.append({
                    'frame': frame_count,
                    'track_id': track_id,
                    'time_ms': time_ms,
                    'class': class_name,
                    'confidence': conf
                })

                colour = getColours(cls)
                cv2.rectangle(frame, (x1, y1), (x2, y2), colour, 2)
                cv2.putText(frame, f"{class_name} {conf:.2f}",
                            (x1, max(y1 - 10, 20)), cv2.FONT_HERSHEY_SIMPLEX,
                            0.6, colour, 2)

    # === БЛОК ПРОГНОЗА ===
    current_minute = int(frame_count * (1000 / fps) // 60000)
    
    if current_minute != last_minute:
        history.append(cars_in_minute)
        cars_in_minute = 0
        
        if len(history) == 30 and model_30 and model_60 and model_120:
            features = [
                history[-1], history[-2], history[-3], 
                history[-5], history[-10], history[-15], 
                history[-20], history[-30], current_minute % 24
            ]
            
            pred_30 = int(model_30.predict([features])[0])
            pred_60 = int(model_60.predict([features])[0])
            pred_120 = int(model_120.predict([features])[0])
            
            predictions.append([current_minute, history[-1], pred_30, pred_60, pred_120])
            
            if len(predictions) % 10 == 0:
                pred_df = pd.DataFrame(predictions, columns=['minute', 'actual', 'pred_30', 'pred_60', 'pred_120'])
                pred_df.to_csv('predictions.csv', index=False)
        
        last_minute = current_minute
    # =================

    if frame_count < 100:
        cv2.imshow('Detection', frame)
        cv2.waitKey(1)

    frame_count += 1

cv2.destroyAllWindows()
videoCap.release()

df = pd.DataFrame(detections)
df.to_csv('detections.csv', index=False)
print(f"Сохранено {len(detections)} записей")
print(f"Уникальных track_id: {df['track_id'].nunique()}")

ModuleNotFoundError: No module named 'joblib'